# Gold test Iceberg view

Notebook nay dung JAR local trong repo, khong dung `spark.jars.packages`/`--packages` va khong download JAR. Cac cell ben duoi xem ca real rows, semantic metadata va Iceberg metadata cho cac bang Gold semantic name moi: `gold_staging.stg_events`, `gold.fact_events`, `gold.fact_sales`, `gold.dim_*`, `gold.daily_*_summary`, va `metadata.*`.


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/home/lyan/Project/BigData/Agent4DA")
os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-17-openjdk-amd64")

def load_env_file(path):
    path = Path(path)
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\'").strip('"'))

load_env_file(PROJECT_ROOT / "envs/minio.env")
load_env_file(PROJECT_ROOT / "envs/postgre.env")
load_env_file(PROJECT_ROOT / "envs/iceberg.env")


In [ ]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" 

In [ ]:
from pyspark.sql import SparkSession

catalog = os.getenv("ICEBERG_CATALOG_NAME", "iceberg_catalog")
jdbc_uri = os.getenv("GOLD_VIEW_ICEBERG_JDBC_URI") or os.getenv(
    "ICEBERG_JDBC_URI",
    "jdbc:postgresql://localhost:5432/agent4da",
).replace("postgres-db", "localhost")
jdbc_schema = os.getenv("ICEBERG_JDBC_SCHEMA", "iceberg")
jdbc_user = os.getenv("ICEBERG_JDBC_USER")
jdbc_password = os.getenv("ICEBERG_JDBC_PASSWORD")
warehouse = os.getenv("GOLD_VIEW_ICEBERG_WAREHOUSE", "s3a://gold/gold/warehouse/")
minio_endpoint = os.getenv("GOLD_VIEW_MINIO_ENDPOINT") or os.getenv(
    "MINIO_ENDPOINT",
    "http://localhost:9000",
).replace("http://minio:9000", "http://localhost:9000")
minio_access_key = os.getenv("MINIO_ACCESS_KEY")
minio_secret_key = os.getenv("MINIO_SECRET_KEY")

required_env = {
    "ICEBERG_JDBC_USER": jdbc_user,
    "ICEBERG_JDBC_PASSWORD": jdbc_password,
    "MINIO_ACCESS_KEY": minio_access_key,
    "MINIO_SECRET_KEY": minio_secret_key,
}
missing = [key for key, value in required_env.items() if not value]
if missing:
    raise ValueError(f"Missing env values: {missing}")

jar_paths = sorted(str(path) for path in (PROJECT_ROOT / "jars").glob("*.jar"))
classpath = ":".join(jar_paths)
jars_csv = ",".join(jar_paths)

spark = (
    SparkSession.builder
    .appName("GoldTestIcebergView")
    .master("local[*]")
    .config("spark.jars", jars_csv)
    .config("spark.driver.extraClassPath", classpath)
    .config("spark.executor.extraClassPath", classpath)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
    .config(f"spark.sql.catalog.{catalog}.uri", jdbc_uri)
    .config(f"spark.sql.catalog.{catalog}.jdbc.user", jdbc_user)
    .config(f"spark.sql.catalog.{catalog}.jdbc.password", jdbc_password)
    .config(f"spark.sql.catalog.{catalog}.jdbc.currentSchema", jdbc_schema)
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


In [ ]:
staging_namespace = "gold_staging"
gold_namespace = "gold"
metadata_namespace = "metadata"

staging_table = f"{catalog}.{staging_namespace}.stg_events"
fact_events_table = f"{catalog}.{gold_namespace}.fact_events"
fact_sales_table = f"{catalog}.{gold_namespace}.fact_sales"
dim_time_table = f"{catalog}.{gold_namespace}.dim_time"
dim_product_table = f"{catalog}.{gold_namespace}.dim_product"
dim_user_table = f"{catalog}.{gold_namespace}.dim_user"
dim_session_table = f"{catalog}.{gold_namespace}.dim_session"
daily_event_summary_table = f"{catalog}.{gold_namespace}.daily_event_summary"
daily_product_summary_table = f"{catalog}.{gold_namespace}.daily_product_summary"
daily_category_summary_table = f"{catalog}.{gold_namespace}.daily_category_summary"
daily_brand_summary_table = f"{catalog}.{gold_namespace}.daily_brand_summary"
table_catalog_table = f"{catalog}.{metadata_namespace}.table_catalog"
column_catalog_table = f"{catalog}.{metadata_namespace}.column_catalog"
metric_catalog_table = f"{catalog}.{metadata_namespace}.metric_catalog"
join_catalog_table = f"{catalog}.{metadata_namespace}.join_catalog"

summary_tables = [
    daily_event_summary_table,
    daily_product_summary_table,
    daily_category_summary_table,
    daily_brand_summary_table,
]

metadata_tables = [
    table_catalog_table,
    column_catalog_table,
    metric_catalog_table,
    join_catalog_table,
]

all_gold_tables = [
    staging_table,
    fact_events_table,
    fact_sales_table,
    dim_time_table,
    dim_product_table,
    dim_user_table,
    dim_session_table,
    *summary_tables,
]

spark.sql(f"SHOW NAMESPACES IN {catalog}").show(truncate=False)
spark.sql(f"SHOW TABLES IN {catalog}.{staging_namespace}").show(truncate=False)
spark.sql(f"SHOW TABLES IN {catalog}.{gold_namespace}").show(truncate=False)
spark.sql(f"SHOW TABLES IN {catalog}.{metadata_namespace}").show(truncate=False)


In [ ]:
spark.table(staging_table).show(20, truncate=False)
spark.sql(f"SELECT count(*) AS staging_rows FROM {staging_table}").show(truncate=False)
spark.sql(f"SELECT event_type, count(*) AS rows FROM {staging_table} GROUP BY event_type ORDER BY event_type").show(truncate=False)


In [ ]:
spark.table(fact_events_table).show(20, truncate=False)
spark.sql(f"SELECT count(*) AS fact_events_rows FROM {fact_events_table}").show(truncate=False)
spark.sql(f"SELECT event_type, count(*) AS rows FROM {fact_events_table} GROUP BY event_type ORDER BY event_type").show(truncate=False)
spark.sql(f"SELECT count(*) AS purchase_rows FROM {fact_events_table} WHERE event_type = 'purchase'").show(truncate=False)


In [ ]:
spark.table(fact_sales_table).show(20, truncate=False)
spark.sql(f"SELECT count(*) AS fact_sales_rows FROM {fact_sales_table}").show(truncate=False)
spark.sql(f"SELECT sum(gross_amount) AS gross_amount FROM {fact_sales_table}").show(truncate=False)


In [ ]:
for table in [dim_time_table, dim_product_table, dim_user_table, dim_session_table]:
    print(f"\n== {table} rows ==")
    spark.table(table).show(20, truncate=False)
    spark.sql(f"SELECT count(*) AS row_count FROM {table}").show(truncate=False)


In [ ]:
for table in summary_tables:
    print(f"\n== {table} rows ==")
    spark.table(table).show(20, truncate=False)
    spark.sql(f"SELECT count(*) AS row_count FROM {table}").show(truncate=False)


In [ ]:
spark.sql(f"SELECT * FROM {daily_product_summary_table} ORDER BY revenue DESC LIMIT 20").show(truncate=False)
spark.sql(f"SELECT * FROM {daily_category_summary_table} ORDER BY revenue DESC LIMIT 20").show(truncate=False)
spark.sql(f"SELECT * FROM {daily_brand_summary_table} ORDER BY revenue DESC LIMIT 20").show(truncate=False)


In [ ]:
spark.table(table_catalog_table).show(100, truncate=False)
spark.table(column_catalog_table).show(100, truncate=False)
spark.table(metric_catalog_table).show(100, truncate=False)
spark.table(join_catalog_table).show(100, truncate=False)


In [ ]:
spark.sql(f"""
SELECT table_name, description, grain
FROM {table_catalog_table}
WHERE recommended_for_agent = true
ORDER BY table_name
""").show(100, truncate=False)

spark.sql(f"""
SELECT column_name, business_name, description, agent_synonyms
FROM {column_catalog_table}
WHERE table_name = 'gold.daily_product_summary'
ORDER BY column_name
""").show(100, truncate=False)

spark.sql(f"""
SELECT metric_name, formula_sql, base_table, example_question
FROM {metric_catalog_table}
ORDER BY metric_name
""").show(100, truncate=False)


In [ ]:
for table in all_gold_tables:
    print(f"\n== {table}.snapshots ==")
    spark.sql(f"SELECT * FROM {table}.snapshots").show(truncate=False)
    print(f"\n== {table}.files ==")
    spark.sql(f"SELECT * FROM {table}.files").show(truncate=False)
